# Experiment 6: Threshold Changes

Starting from a 2-of-3 group, we increase the threshold to 3-of-3 (stricter
security), then decrease it back to 2-of-3 (by having one participant reveal
their share). The group public key is preserved throughout.

**Related:** [Tutorial 5: DKG Ceremony](tutorial-05-dkg.ipynb), Guide Chapter 8 (Advanced Operations)

In [ ]:
import sys
sys.path.insert(0, "../src")

from frost import Participant, Aggregator, Scalar, Point, G, Q
from frost.tagged_hash import tagged_hash
from frost.lagrange import lagrange_coefficient

# DKG setup (2-of-3)
threshold = 2
n = 3
participants = [Participant(index=i, threshold=threshold, participants=n) for i in range(1, n + 1)]

for p in participants:
    p.init_keygen()
for p in participants:
    p.generate_shares()
for receiver in participants:
    other_shares = tuple(
        sender.shares[receiver.index - 1]
        for sender in participants if sender.index != receiver.index
    )
    receiver.aggregate_shares(other_shares)
for p in participants:
    other_commitments = tuple(
        other.coefficient_commitments[0]
        for other in participants if other.index != p.index
    )
    p.derive_public_key(other_commitments)
for p in participants:
    other_ccs = tuple(
        other.coefficient_commitments
        for other in participants if other.index != p.index
    )
    p.derive_group_commitments(other_ccs)

public_key = participants[0].public_key
original_pk_x = public_key.x
print(f"Original: {threshold}-of-{n}, public key x = {original_pk_x}")


## Threshold Increase: 2-of-3 to 3-of-3

Each participant generates a new polynomial of degree `new_threshold - 2` = 1
(a linear polynomial). They evaluate it at all participant indexes and exchange
the evaluations. Each participant then adjusts their aggregate share.

The formula: `f'(i) = f(i) + i · ∑ⱼ g_j(i)` (summing over all participants
j = 1..n), where `g_j` are the new polynomials. The multiplication by i
preserves the constant term: f'(0) = f(0).

In [ ]:
new_threshold = 3

# Each participant generates a threshold increase polynomial
for p in participants:
    p.init_threshold_increase(new_threshold)

# Each participant generates shares (evaluations of the new polynomial)
for p in participants:
    p.generate_shares()

# Each participant increases their threshold using their own and others' shares
for p in participants:
    other_shares = tuple(
        other.shares[p.index - 1]
        for other in participants if other.index != p.index
    )
    p.increase_threshold(other_shares)

print(f"Threshold increased to {new_threshold}")
print(f"Public key preserved: {public_key.x == original_pk_x}")

# Verify with Lagrange reconstruction: 3-of-3 should reconstruct the secret
l1 = int(lagrange_coefficient((1, 2, 3), 1))
l2 = int(lagrange_coefficient((1, 2, 3), 2))
l3 = int(lagrange_coefficient((1, 2, 3), 3))
secret = (participants[0].aggregate_share * l1 + participants[1].aggregate_share * l2 + participants[2].aggregate_share * l3) % Q
print(f"3-of-3 reconstructs to group key: {secret * G == public_key}")

## Test: 2-of-3 Should No Longer Reconstruct

With threshold now 3, Lagrange interpolation with only 2 points can't
reconstruct a degree-2 polynomial.


In [ ]:
# 2-of-3 should NOT reconstruct the correct secret
l1_2 = int(lagrange_coefficient((1, 2), 1))
l2_2 = int(lagrange_coefficient((1, 2), 2))
secret_2 = (participants[0].aggregate_share * l1_2 + participants[1].aggregate_share * l2_2) % Q
print(f"2-of-3 reconstructs correctly? {secret_2 * G == public_key}")
print("Insufficient shares cannot recover the secret: the polynomial has degree 2.")

## Test: Signing with 3-of-3

Update group commitments for the new threshold degree and sign.


In [ ]:
from frost.threshold import derive_coefficient_commitments

# Derive new group commitments for degree-2 polynomial
pvs = tuple(int(p.aggregate_share) * G for p in participants)
indexes = tuple(p.index for p in participants)
new_gc = derive_coefficient_commitments(pvs, indexes)
for p in participants:
    p.group_commitments = new_gc

print(f"New group commitments: {len(new_gc)} coefficients (was 2, now 3)")

# Sign with all 3 participants
message = b"Threshold test"
signers = [participants[0], participants[1], participants[2]]
signer_indexes = tuple(p.index for p in signers)

for signer in signers:
    signer.generate_nonce_pair()

ncp = [(Point(), Point())] * n
for signer in signers:
    ncp[signer.index - 1] = signer.nonce_commitment_pair
ncp = tuple(ncp)

shares = [signer.sign(message, ncp, signer_indexes) for signer in signers]
agg = Aggregator(public_key, message, ncp, signer_indexes)
sig_hex = agg.signature(tuple(shares))

# BIP340 verification
sig_bytes = bytes.fromhex(sig_hex)
R_x = int.from_bytes(sig_bytes[:32], "big")
s = int.from_bytes(sig_bytes[32:], "big")
R = Point.lift_x(R_x)
pk = Point.lift_x(public_key.x)
e_bytes = tagged_hash("BIP0340/challenge", sig_bytes[:32] + pk.to_bytes_xonly() + message)
e = int.from_bytes(e_bytes, "big") % Q
valid_3 = s * G == R + (e * pk)
print(f"Valid with 3 signers (threshold 3)? {valid_3}")
print(f"Public key still matches: {public_key.x == original_pk_x}")


## Threshold Decrease: 3-of-3 Back to 2-of-3

One participant publicly reveals their share. This is destructive: the
revealing participant permanently loses their share. The others use
`decrement_threshold` to compute their new shares on a lower-degree polynomial.


In [ ]:
# P3 reveals their share
revealed_share = participants[2].aggregate_share
revealed_index = 3
print(f"P3 reveals share: {revealed_share}")
print("P3 can no longer participate in the group.")

# P1 and P2 decrement their thresholds
for p in [participants[0], participants[1]]:
    p.decrement_threshold(revealed_share, revealed_index)

print(f"\nP1 new threshold: {participants[0].threshold}")
print(f"P2 new threshold: {participants[1].threshold}")
print(f"Public key preserved: {public_key.x == original_pk_x}")


## Test: Signing with 2 Should Succeed Again


In [ ]:
message = b"Back to 2-of-3"
signers = [participants[0], participants[1]]
signer_indexes = tuple(p.index for p in signers)

for signer in signers:
    signer.generate_nonce_pair()

ncp = [(Point(), Point())] * n
for signer in signers:
    ncp[signer.index - 1] = signer.nonce_commitment_pair
ncp = tuple(ncp)

shares = [signer.sign(message, ncp, signer_indexes) for signer in signers]
agg = Aggregator(public_key, message, ncp, signer_indexes)
sig_hex = agg.signature(tuple(shares))

# BIP340 verification
sig_bytes = bytes.fromhex(sig_hex)
R_x = int.from_bytes(sig_bytes[:32], "big")
s = int.from_bytes(sig_bytes[32:], "big")
R = Point.lift_x(R_x)
pk = Point.lift_x(public_key.x)
e_bytes = tagged_hash("BIP0340/challenge", sig_bytes[:32] + pk.to_bytes_xonly() + message)
e = int.from_bytes(e_bytes, "big") % Q
valid_final = s * G == R + (e * pk)
print(f"Valid with 2 signers (threshold 2 again)? {valid_final}")
print(f"Public key preserved throughout: {public_key.x == original_pk_x}")


## Results

| Step | Threshold | Sign with 2 | Sign with 3 | Public Key |
|------|-----------|-------------|-------------|------------|
| Original | 2-of-3 | Valid | Valid | Same |
| After increase | 3-of-3 | Invalid | Valid | Same |
| After decrease | 2-of-3 | Valid | N/A (group is now 2-of-2, P3 disenrolled) | Same |

**Tradeoffs:**
- **Increase**: more security (more signers required), less availability. No
  information is destroyed.
- **Decrease**: more availability (fewer signers required), less security. One
  share is publicly revealed, permanently removing that participant.

The group public key, which is f(0)·G, is preserved through both operations
because both transformations preserve the polynomial's constant term.

## Next Steps

- What if you increase by more than 1 (e.g., 2-of-3 to 4-of-5)?
- The Vandermonde matrix inversion in `derive_coefficient_commitments` is used
  to update group commitments after both operations.
- See Guide Chapter 8 (Advanced Operations) for the full math.